In [5]:
import numpy as np
import pandas as pd
import ast # Automated String Evaluation to parse the JSON columns
import pickle

# Load the datasets
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

# Look at the first few rows of the data
print("Movies shape:", movies.shape)
print("Credits shape:", credits.shape)
movies.head(1)

Movies shape: (4803, 20)
Credits shape: (4803, 4)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [6]:
# Merge the dataframes on the 'title' column
movies = movies.merge(credits, on='title')

# Verify the new shape (it should have 23 columns now)
print("Merged shape:", movies.shape)
movies.head(1)

Merged shape: (4809, 23)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [7]:
# Keep only the columns needed for content recommendation
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

# Check for missing values and drop them
print("Missing values per column:\n", movies.isnull().sum())
movies.dropna(inplace=True)

movies.head()

Missing values per column:
 movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64


,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [8]:
# 1. Helper function to extract all names from genres and keywords
def convert_json_to_list(obj):
    names_list = []
    for i in ast.literal_eval(obj):
        names_list.append(i['name'])
    return names_list

# Apply the helper function to genres and keywords
movies['genres'] = movies['genres'].apply(convert_json_to_list)
movies['keywords'] = movies['keywords'].apply(convert_json_to_list)

# 2. Helper function to extract only the top 3 actors from the cast
def convert_cast(obj):
    cast_list = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            cast_list.append(i['name'])
            counter += 1
        else:
            break
    return cast_list

# Apply to the cast column
movies['cast'] = movies['cast'].apply(convert_cast)

# 3. Helper function to extract ONLY the Director from the crew
def fetch_director(obj):
    director_list = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            director_list.append(i['name'])
            break
    return director_list

# Apply to the crew column
movies['crew'] = movies['crew'].apply(fetch_director)

# Check our transformation progress
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


In [9]:
# Convert the overview string into a list of words split by spaces
movies['overview'] = movies['overview'].apply(lambda x: str(x).split())

movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


In [10]:
# 1. Remove spaces from all elements inside our lists
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ", "") for i in x])

# 2. Combine all 5 metadata columns into one single 'tags' column
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# 3. Create a brand-new, clean dataframe with only the columns we actually need now
new_df = movies[['movie_id', 'title', 'tags']]

new_df.head()

,movie_id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [11]:
# Convert the list of words back into a single space-separated string
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

# Convert everything to lowercase
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

# Let's inspect the final finalized data structure!
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


In [12]:
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

# Helper function to stem each word in the tags string
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

# Apply stemming to the tags column
new_df['tags'] = new_df['tags'].apply(stem)

# Print a preview of the first movie's tags to see the changes
print(new_df['tags'][0][:100])

in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom 


In [13]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')

# Transform the text tags into numerical vectors
vectors = cv.fit_transform(new_df['tags']).toarray()

# Check the shape (it should be 4803 movies by 5000 word columns)
print("Vectors shape:", vectors.shape)

Vectors shape: (4806, 5000)


In [14]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate the similarity score of every movie against every other movie
similarity = cosine_similarity(vectors)

# Check the matrix shape (it will be 4803 x 4803)
print("Similarity Matrix Shape:", similarity.shape)

Similarity Matrix Shape: (4806, 4806)


In [17]:
def recommend(movie):
    try:
        # 1. Find the index of the given movie
        movie_index = new_df[new_df['title'] == movie].index[0]
        
        # 2. Get the similarity vector for that movie
        distances = similarity[movie_index]
        
        # 3. Sort the distances but keep the original index tracking intact
        # We take [1:6] because index 0 is the movie itself (100% match)
        movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
        
        # 4. Print the top 5 recommended movie titles
        print(f"Recommendations for '{movie}':\n")
        print("-" * 30)
        for i in movies_list:
            print(new_df.iloc[i[0]].title)
            
    except IndexError:
        print(f"Movie '{movie}' not found in the dataset. Please check spelling!")

# Test your brand new recommendation engine!
recommend('Avatar')

Recommendations for 'Avatar':

------------------------------
Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [18]:
import pickle

# We convert the dataframe to a dictionary before pickling 
# because it is much easier for a web server to read later!
pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))

# Save the massive similarity matrix
pickle.dump(similarity, open('similarity.pkl', 'wb'))

print("Model successfully exported!")

Model successfully exported!
